In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '2'

### Set Unit Cell

In [5]:
structure.cell.length_a = 5.431

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-si-sepd', destination='data')

Getting data...


Data 'meas-si-sepd': Si, SEPD (Argonne)


✅ Data 'meas-si-sepd' already present at '../../../data/meas-si-sepd.xye'. Keeping existing.


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='sepd',
    data_path=data_path,
    beam_mode='time-of-flight',
)

### Set Instrument

In [9]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = -10.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quadratic = -1.54

### Set Peak Profile

In [10]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


In [11]:
expt.peak.type = 'jorgensen-von-dreele'

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    
   • broad_lorentz_size_l=0.0                                                                                                     
   • broad_lorentz_strain_l=0.0                                                                                                   


Peak profile type for experiment 'sepd' changed to


jorgensen-von-dreele


In [12]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_gauss_sigma_2 = 0.0
expt.peak.broad_lorentz_gamma_0 = 0.0
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.broad_lorentz_gamma_2 = 0.0
expt.peak.rise_alpha_0 = 0.0
expt.peak.rise_alpha_1 = 0.5971
expt.peak.decay_beta_0 = 0.0408
expt.peak.decay_beta_1 = 0.0123

In [13]:
expt.peak.cutoff_fwhm = 8.2

### Set Background

In [14]:
expt.background.auto_estimate()

### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='si', scale=600.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [16]:
project = Project(name='si_sepd')

### Add Structure

In [17]:
project.structures.add(structure)

### Add Experiment

In [18]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [19]:
project.display.structure(struct_name='si')

Structure 🧩 'si' (Atom view type: 'covalent')


### Display Pattern

In [20]:
project.display.pattern(expt_name='sepd')
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 1/4

Set parameters to be refined.

In [21]:
structure.cell.length_a.free = True

expt.linked_structures['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [22]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43100,,-inf,inf,Å
2,sepd,linked_structure,si,scale,600.00000,,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-10.00000,,-inf,inf,μs


#### Run Fitting

In [23]:
project.analysis.minimizer.type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


In [24]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),13.43
4,📏 Goodness-of-fit (reduced χ²),6.91
5,"📏 R-factor (Rf, %)",11.58
6,"📏 R-factor squared (Rf², %)",6.18
7,"📏 Weighted R-factor (wR, %)",5.09


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4310,5.4308,0.0001,0.00 % ↓
2,sepd,linked_structure,si,scale,,600.0000,368.2037,0.9626,38.63 % ↓
3,sepd,instrument,,d_to_tof_offset,μs,-10.0000,-8.3129,0.0760,16.87 % ↓


#### Display Pattern

In [25]:
project.display.pattern(expt_name='sepd')

In [26]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 2/4

Set more parameters to be refined.

In [27]:
for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [28]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43082,0.00006,-inf,inf,Å
2,sepd,linked_structure,si,scale,368.20374,0.96260,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-8.31294,0.07597,-inf,inf,μs
4,sepd,background,1,intensity,213.55062,,-inf,inf,
5,sepd,background,2,intensity,117.61669,,-inf,inf,
6,sepd,background,3,intensity,147.70005,,-inf,inf,
7,sepd,background,4,intensity,122.26237,,-inf,inf,
8,sepd,background,5,intensity,163.04903,,-inf,inf,
9,sepd,background,6,intensity,124.58762,,-inf,inf,
10,sepd,background,7,intensity,120.30738,,-inf,inf,


#### Run Fitting

In [29]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),28.25
4,📏 Goodness-of-fit (reduced χ²),3.71
5,"📏 R-factor (Rf, %)",8.29
6,"📏 R-factor squared (Rf², %)",4.20
7,"📏 Weighted R-factor (wR, %)",3.13


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4308,5.4309,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,368.2037,379.1034,0.7275,2.96 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,-8.3129,-8.4374,0.0541,1.50 % ↑
4,sepd,background,1,intensity,,213.5506,203.7862,0.4110,4.57 % ↓
5,sepd,background,2,intensity,,117.6167,103.7428,0.4466,11.80 % ↓
6,sepd,background,3,intensity,,147.7001,125.1144,0.8726,15.29 % ↓
7,sepd,background,4,intensity,,122.2624,119.8186,0.9651,2.00 % ↓
8,sepd,background,5,intensity,,163.0490,127.4725,2.7426,21.82 % ↓
9,sepd,background,6,intensity,,124.5876,123.1576,1.6805,1.15 % ↓
10,sepd,background,7,intensity,,120.3074,121.7270,1.7645,1.18 % ↑


#### Display Pattern

In [30]:
project.display.pattern(expt_name='sepd')

In [31]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 3/4

Fix background points.

In [32]:
for point in expt.background:
    point.intensity.free = False

Set more parameters to be refined.

In [33]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True

Show free parameters after selection.

In [34]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43092,0.00004,-inf,inf,Å
2,sepd,linked_structure,si,scale,379.10344,0.72748,-inf,inf,
3,sepd,peak,,broad_lorentz_gamma_1,2.54890,,-inf,inf,μs/Å
4,sepd,peak,,broad_gauss_sigma_0,3.01480,,-inf,inf,μs²
5,sepd,peak,,broad_gauss_sigma_1,33.34510,,-inf,inf,μs/Å
6,sepd,instrument,,d_to_tof_offset,-8.43737,0.05410,-inf,inf,μs


#### Run Fitting

In [35]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),10.20
4,📏 Goodness-of-fit (reduced χ²),3.63
5,"📏 R-factor (Rf, %)",8.33
6,"📏 R-factor squared (Rf², %)",4.27
7,"📏 Weighted R-factor (wR, %)",3.14


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4309,5.4309,0.0000,0.00 % ↓
2,sepd,linked_structure,si,scale,,379.1034,378.7425,0.7565,0.10 % ↓
3,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.5489,2.2595,0.0734,11.35 % ↓
4,sepd,peak,,broad_gauss_sigma_0,μs²,3.0148,6.1390,0.3670,103.63 % ↑
5,sepd,peak,,broad_gauss_sigma_1,μs/Å,33.3451,32.7143,0.6814,1.89 % ↓
6,sepd,instrument,,d_to_tof_offset,μs,-8.4374,-8.3741,0.0567,0.75 % ↓


#### Display Pattern

In [36]:
project.display.pattern(expt_name='sepd')

In [37]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 4/4

Set more parameters to be refined.

In [38]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True

Show free parameters after selection.

In [39]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43088,0.00004,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²
3,sepd,linked_structure,si,scale,378.74250,0.75650,-inf,inf,
4,sepd,peak,,decay_beta_0,0.04080,,-inf,inf,μs
5,sepd,peak,,decay_beta_1,0.01230,,-inf,inf,μs/Å
6,sepd,peak,,broad_lorentz_gamma_1,2.25952,0.07343,-inf,inf,μs/Å
7,sepd,peak,,broad_gauss_sigma_0,6.13903,0.36696,-inf,inf,μs²
8,sepd,peak,,broad_gauss_sigma_1,32.71430,0.68138,-inf,inf,μs/Å
9,sepd,instrument,,d_to_tof_offset,-8.37410,0.05669,-inf,inf,μs


#### Run Fitting

In [40]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),15.67
4,📏 Goodness-of-fit (reduced χ²),3.60
5,"📏 R-factor (Rf, %)",8.21
6,"📏 R-factor squared (Rf², %)",4.12
7,"📏 Weighted R-factor (wR, %)",2.93


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4309,5.4309,0.0001,0.00 % ↑
2,si,atom_site,Si,adp_iso,Å²,0.5000,0.5222,0.0036,4.44 % ↑
3,sepd,linked_structure,si,scale,,378.7425,383.0852,1.0346,1.15 % ↑
4,sepd,peak,,decay_beta_0,μs,0.0408,0.0406,0.0002,0.45 % ↓
5,sepd,peak,,decay_beta_1,μs/Å,0.0123,0.0124,0.0002,0.94 % ↑
6,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.2595,2.2264,0.0799,1.47 % ↓
7,sepd,peak,,broad_gauss_sigma_0,μs²,6.1390,5.6439,0.4303,8.06 % ↓
8,sepd,peak,,broad_gauss_sigma_1,μs/Å,32.7143,33.0053,0.6995,0.89 % ↑
9,sepd,instrument,,d_to_tof_offset,μs,-8.3741,-8.4008,0.0811,0.32 % ↑


#### Display Correlations

In [41]:
project.display.fit.correlations()

#### Display Pattern

In [42]:
project.display.pattern(expt_name='sepd')

In [43]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [44]:
project.display.pattern(expt_name='sepd', x='d_spacing')

## 💾 Save Project

In [45]:
project.save_as(dir_path='projects/refine-si-sepd')

Saving project 📦 'si_sepd' to '../../../projects/refine-si-sepd'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 si.edi


├── 📁 experiments/


│   └── 📄 sepd.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 si_sepd.html
